# Assignment 2: Neural Network - Time Series - ECG

Author:  
Ismail Mohammed

## Imports

In [19]:
import numpy as np
import pandas as pd
import ast
import matplotlib.pyplot as plt
from utility_functions import load_raw_data, aggregate_diagnostic
from pandas import read_csv
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, LSTM, Dropout
from keras.callbacks import EarlyStopping
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import mean_absolute_error, r2_score

## Load data

In [20]:
path = 'raw_data/'
sampling_rate=100

# load and convert annotation data
Y = pd.read_csv(path+'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))

# Load raw signal data
X = load_raw_data(Y, sampling_rate, path)

In [22]:
X.shape

(21799, 1000, 12)

In [23]:
# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(path+'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

# Apply diagnostic superclass
Y['diagnostic_subclass'] = Y.scp_codes.apply(lambda x: aggregate_diagnostic(x, agg_df))

In [24]:
Y.shape

(21799, 28)

## Task 1: This is the title of task 1

Use markdown cells like this one to include:
- Discussion points.
- References to specific sources of code that you might have used to solve the assignment.
- General commentas and explanations about your solution.

In [25]:
mlb = MultiLabelBinarizer()
y_encoded = mlb.fit_transform(Y['diagnostic_subclass'])

In [26]:
n_outputs = len(mlb.classes_)
print(f"Unique subclasses: {n_outputs}")
print(f"Classes: {mlb.classes_}")

Unique subclasses: 23
Classes: ['AMI' 'CLBBB' 'CRBBB' 'ILBBB' 'IMI' 'IRBBB' 'ISCA' 'ISCI' 'ISC_' 'IVCD'
 'LAFB/LPFB' 'LAO/LAE' 'LMI' 'LVH' 'NORM' 'NST_' 'PMI' 'RAO/RAE' 'RVH'
 'SEHYP' 'STTC' 'WPW' '_AVB']


In [27]:
train_mask = Y.strat_fold <= 8
val_mask = Y.strat_fold == 9

# Plocka ut X (signaler) och Y (encoded labels) för träning
X_train = X[train_mask]
y_train = y_encoded[train_mask]

# Plocka ut X och Y för validering
X_val = X[val_mask]
y_val = y_encoded[val_mask]

print(f"Träningsdata shape: {X_train.shape}")
print(f"Valideringsdata shape: {X_val.shape}")

Träningsdata shape: (17418, 1000, 12)
Valideringsdata shape: (2183, 1000, 12)


In [ ]:
# Parametrar för vår data (om vi använder 100Hz = 1000 punkter per 10 sekunder)
n_timesteps = 1000
n_features = 12
n_outputs = 23

model = Sequential()
model.add(LSTM(100, input_shape=(n_timesteps, n_features)))

model.add(Dropout(0.5))

# Output-lager. Eftersom en patient kan ha FLERA sjukdomar samtidigt 
# (Multi-label) MÅSTE vi ha 'sigmoid', inte 'softmax'.
model.add(Dense(n_outputs, activation='sigmoid'))

# Kompilera modellen. För multilabel classification använder vi binary_crossentropy
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=[tf.keras.metrics.AUC(multi_label=True)])

model.summary()

c:\Users\isse_\miniconda3\envs\ekg_lab\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 100)            │        45,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 23)             │         2,323 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 47,523 (185.64 KB)

 Trainable params: 47,523 (185.64 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stop = EarlyStopping(
    monitor='loss', 
    mode='min', 
    patience=5, 
    restore_best_weights=True
)

history = model.fit(
    X_train, 
    y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

Epoch 1/30
545/545 ━━━━━━━━━━━━━━━━━━━━ 94s 172ms/step - auc_2: 0.5464 - loss: 0.2211 - val_auc_2: 0.5034 - val_loss: 0.1897
Epoch 2/30
545/545 ━━━━━━━━━━━━━━━━━━━━ 117s 214ms/step - auc_2: 0.5128 - loss: 0.1888 - val_auc_2: 0.5107 - val_loss: 0.1790
Epoch 3/30
545/545 ━━━━━━━━━━━━━━━━━━━━ 118s 216ms/step - auc_2: 0.5040 - loss: 0.1848 - val_auc_2: 0.5093 - val_loss: 0.1785
Epoch 4/30
545/545 ━━━━━━━━━━━━━━━━━━━━ 115s 212ms/step - auc_2: 0.5079 - loss: 0.1835 - val_auc_2: 0.5102 - val_loss: 0.1784
Epoch 5/30
545/545 ━━━━━━━━━━━━━━━━━━━━ 117s 215ms/step - auc_2: 0.5055 - loss: 0.1830 - val_auc_2: 0.5235 - val_loss: 0.1780
Epoch 6/30
545/545 ━━━━━━━━━━━━━━━━━━━━ 119s 219ms/step - auc_2: 0.5072 - loss: 0.1825 - val_auc_2: 0.5287 - val_loss: 0.1780
Epoch 7/30
545/545 ━━━━━━━━━━━━━━━━━━━━ 118s 216ms/step - auc_2: 0.5046 - loss: 0.1823 - val_auc_2: 0.5322 - val_loss: 0.1778
Epoch 8/30
545/545 ━━━━━━━━━━━━━━━━━━━━ 116s 212ms/step - auc_2: 0.5173 - loss: 0.1817 - val_auc_2: 0.5215 - val_loss: 

## Task 2: This is the title of task 2

This section should contain the solution of task 2.

## Results and Discussion

This section should contain:
- Results.
- Summary of best model performance:
    - Name of best model file as saved in /models.
    - Relevant scores such as: accuracy, precision, recall, F1-score, etc.
- Key discussion points.